**Importer les libraries pertinentes**

In [2]:
import pandas as pd
from os import listdir
from os import path
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

**Entraîner les modèles et sortir les résultats**

In [3]:
folder = '../1-data/training_datasets/'
datasets = listdir(folder)
datasets

['train_dataset_10pc.xlsx',
 'train_dataset_20pc.xlsx',
 'train_dataset_30pc.xlsx',
 'train_dataset_40pc.xlsx',
 'train_dataset_50pc.xlsx',
 'train_dataset_60pc.xlsx',
 'train_dataset_70pc.xlsx',
 'train_dataset_80pc.xlsx',
 'train_dataset_90pc.xlsx']

In [20]:
folder = '../1-data/training_datasets/'
datasets = listdir(folder)
datasets = [
 'train_dataset_50pc.xlsx',
 'train_dataset_70pc.xlsx',
]

for file in datasets:
    ratio = file[14:-7]
    report = ""

    # Pour information, la section qui suit a été écrite avec l'aide de ChatGPT
    df = pd.read_excel(path.join(folder, file))
    X = df['text_post'].astype(str)
    y = df['category']


    # 5-fold cross-validation
    stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Valeurs du paramètre k à tester pour les classifieurs knn
    k_values = [1, 2, 3, 4, 5]

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 1000, 1500, 2000, 3000, 4000, 5000]

    ngram_values = (1, 2)

    # Antidictionnaire personnalisé
    with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
        custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
        custom_stopwords += list(ENGLISH_STOP_WORDS)

    # Créer un pipeline d'expérimentation pour le classifieur Naïve Bayes
    for n_features in n_features_values:
            pipeline_nb = Pipeline([
                ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=n_features)),
                ('nb', MultinomialNB())
            ])

            # Validation croisée 
            nb_scores = cross_val_score(pipeline_nb, X, y, cv=stratified_kfold, scoring='accuracy')
            nb_predictions = cross_val_predict(pipeline_nb, X, y, cv=stratified_kfold)

            # Performances (rappel, précision, score F)
            parametres = f"% données Incels : {ratio}, Algorithme : Naive Bayes, Features={n_features}, Accuracy = {nb_scores.mean()}"
            results = classification_report(y, nb_predictions)
            print(parametres)
            print(results)

            report += parametres
            report += '\n'
            report += results
            report += '---\n'


    # Tester différents paramètres pour le classifieur kNN
    for k in k_values:
        for n_features in n_features_values:
            # Créer un pipeline avec un classifieur kNN
            pipeline_knn = Pipeline([
                ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=n_features)),
                ('knn', KNeighborsClassifier(n_neighbors=k))
            ])

            # Utiliser la validation croisée avec le classifieur kNN
            knn_scores = cross_val_score(pipeline_knn, X, y, cv=stratified_kfold, scoring='accuracy')
            knn_predictions = cross_val_predict(pipeline_knn, X, y, cv=stratified_kfold)

            # Performances moyennes sur les plis pour kNN
            parametres = f"% données Incels : {ratio}, Algorithme : kNN (k={k}), Features={n_features}, Accuracy = {knn_scores.mean()}"
            results = classification_report(y, knn_predictions)
            print(parametres)
            print(results)

            report += parametres
            report += '\n'
            report +=  results
            report += '---\n'

    with open(f'../3-results_training/results_training_{ratio}pc.txt', 'w', encoding='utf-8') as f:
         f.write(report)

% données Incels : 50, Algorithme : Naive Bayes, Features=100, Accuracy = 0.5501400000000001
              precision    recall  f1-score   support

       incel       0.54      0.71      0.61     25000
     neutral       0.57      0.39      0.47     25000

    accuracy                           0.55     50000
   macro avg       0.56      0.55      0.54     50000
weighted avg       0.56      0.55      0.54     50000

% données Incels : 50, Algorithme : Naive Bayes, Features=250, Accuracy = 0.5969599999999999
              precision    recall  f1-score   support

       incel       0.58      0.70      0.64     25000
     neutral       0.62      0.49      0.55     25000

    accuracy                           0.60     50000
   macro avg       0.60      0.60      0.59     50000
weighted avg       0.60      0.60      0.59     50000

% données Incels : 50, Algorithme : Naive Bayes, Features=500, Accuracy = 0.63594
              precision    recall  f1-score   support

       incel       0.62

**Tester le système retenu sur de nouvelles données**

*On reproduit l'apprentissage pour le système retenu uniquement*

In [4]:
file_train = '../1-data/training_datasets/train_dataset_40pc.xlsx'

df_train = pd.read_excel(file_train)
df_train

,id_post,date_post,subreddit,author,text_post,category
0,i1ihqm5,2022,ForeverAlone,borgwardB,find a hobby you're interested in that has a l...,incel
1,hklyo5r,2021,ForeverAlone,Polaris022,"Well, you can be realistic about your looks an...",incel
2,dnr6u54,2017,ForeverAlone,changeIsTheWay,Oh yeah. You're the chick who I commented on a...,incel
3,dldhg5i,2017,Incels,Pr_Revolving,Honestly this is beyond disturbing.,incel
4,g3ka3n0,2020,IncelsWithoutHate,Intelligent_Badger_1,Is it really true that americans dislike sharp...,incel
...,...,...,...,...,...,...
49995,esg0s29,2019,financialindependence,sulli_p,"Okay I’m fairly new to investing, what about s...",neutral
49996,fzxnufo,2020,realtors,VaBeachRealtor,How is it not? \n\nJust because it comes with ...,neutral
49997,hv2cgip,2022,exjw,DuchessBolo,Absolutely!,neutral
49998,hv2cvjs,2022,tuckedinfishies,thousandlotuspetals_,Mine does this all the time On her piece of wo...,neutral


In [5]:
# Système retenu : Naive Bayes ; 40% de données Incels ; 5 000 traits discriminants
X_train = df_train['text_post'].astype(str)
y_train = df_train['category']

ngram_values = (1, 2)

# Antidictionnaire personnalisé
with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
    custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
    custom_stopwords += list(ENGLISH_STOP_WORDS)

pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=15000)),
    ('nb', MultinomialNB())
])

pipeline_nb.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=15000, ngram_range=(1, 2),
                                 stop_words=['\ufeff&gt', 'a', 'â', 'ä', 'ã',
                                             'å', 'ªã', 'about', 'actually',
                                             'æ', 'again', 'against', "ain't",
                                             'all', 'allow', 'allows', 'almost',
                                             'alone', 'along', 'already',
                                             'also', 'although', 'always', 'am',
                                             'among', 'amongst', 'amp', 'an',
                                             'and', 'another', ...])),
                ('nb', MultinomialNB())])

In [17]:
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nb_predictions = cross_val_predict(pipeline_nb, X_train, y_train, cv=stratified_kfold)

# Performances en phase d'apprentissage du modèle retenu (rappel, précision, score F)
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_train, nb_predictions)
precision_nb = precision_score(y_train, nb_predictions, average='macro')
recall_nb = recall_score(y_train, nb_predictions, average='macro')
f1_nb = f1_score(y_train, nb_predictions, average='macro')

df = [
    {
        'Phase': 'Apprentissage',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
]

print(classification_report(y_train, nb_predictions))

              precision    recall  f1-score   support

       incel       0.80      0.62      0.70     20000
     neutral       0.78      0.89      0.83     30000

    accuracy                           0.78     50000
   macro avg       0.79      0.76      0.77     50000
weighted avg       0.79      0.78      0.78     50000



*On teste sur de nouvelles données*

In [18]:
file_test = '../1-data/test_dataset_10pc.xlsx'

df_test = pd.read_excel(file_test)
df_test

,date_post,id_post,author,subreddit,text_post,category
0,2021,hewyqir,KallystoNyx,FreeKarma4U,Bang!,neutral
1,2022,hv2cm15,Hopeful-Talk-1556,canada,I mean this is why you often get downvoted in ...,neutral
2,2021,hewyfkk,naughtyrpfan,HentaiAndRoleplayy,You playing m or f?,neutral
3,2016,d1l6201,greigmusic,IAmA,"Lol, google black pudding and your view will c...",neutral
4,2020,fzxnd3h,NorCalifornioAH,MapPorn,That's more a matter of Combined Statistical A...,neutral
...,...,...,...,...,...,...
9995,2021,hgdqoim,throwamay555,ForeverAlone,who said you were ugly?,incel
9996,2021,hmqkhx0,Admirable_Elk_965,ForeverAlone,Yeah I see this too when I walk out of class. ...,incel
9997,2022,iw0zptn,Resident_Setting3884,ForeverAlone,"this is exactly how i feel, im a cab driver, t...",incel
9998,2017,dfiq9mw,CALLMEMANTEE,Incels,I mean I'm not ok with it because its depressi...,incel


*Vérifier qu'aucun donnée test n'était contenue dans les données d'appentissage*

In [19]:
validate = pd.concat([df_train, df_test])

validate[validate.duplicated(subset='id_post')]

,id_post,date_post,subreddit,author,text_post,category


*Appliquer le classifieur aux nouvelles données*

In [20]:
X_test = df_test['text_post'].astype(str)
y_test = df_test['category']

y_pred_nb = pipeline_nb.predict(X_test)

In [21]:
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb, average='macro')
recall_nb = recall_score(y_test, y_pred_nb, average='macro')
f1_nb = f1_score(y_test, y_pred_nb, average='macro')

df.append(
    {
        'Phase': 'Test',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
)

pd.set_option("display.precision", 4)
df = pd.DataFrame(df)
df.to_csv('../3-results/results_test.csv', index=False)
df

,Phase,Précision,Rappel,Mesure F,Exactitude
0,Apprentissage,0.7878,0.7573,0.7651,0.7846
1,Test,0.6753,0.7566,0.7033,0.8675


In [22]:
accuracy_nb

0.8675

In [12]:
precision_nb

0.6753170186405374

In [13]:
recall_nb

0.7566111111111111

In [14]:
f1_nb

0.7033224238831136

In [15]:
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

       incel       0.40      0.62      0.48      1000
     neutral       0.95      0.90      0.92      9000

    accuracy                           0.87     10000
   macro avg       0.68      0.76      0.70     10000
weighted avg       0.90      0.87      0.88     10000



In [4]:
import pandas as pd
results = pd.read_csv('../3-results/results_training.csv')
results.sort_values(by='f_score', ascending=False)

,ratio_incels,algorithm,no_features,accuracy,precision,recall,f_score
205,40,Naive Bayes,15000,0.78464,0.79,0.76,0.77
204,40,Naive Bayes,10000,0.78258,0.79,0.76,0.76
178,40,Naive Bayes,5000,0.77686,0.78,0.75,0.76
177,40,Naive Bayes,4000,0.77258,0.78,0.74,0.75
271,50,Naive Bayes,15000,0.74684,0.75,0.75,0.75
...,...,...,...,...,...,...,...
51,10,kNN (k=2),250,0.31376,0.50,0.51,0.29
54,10,kNN (k=3),100,0.29392,0.48,0.47,0.28
116,20,kNN (k=2),100,0.28264,0.50,0.50,0.28
46,10,kNN (k=1),100,0.27790,0.48,0.47,0.26
